# Notebook_2_SQL_Gold_Load — Gold KPIs
### O&G Rig Operations Intelligence Platform
**Layer:** Silver → Gold  
**Run after:** Notebook_1_SQL_Silver_Load  
**Strategy:** Watermark-based incremental MERGE — only affected months recomputed  

| # | Table | Grain |
|---|---|---|
| 1 | gold.rig_performance_summary | rig_id × month_year |
| 2 | gold.equipment_reliability | equipment_type × region_name × month_year |
| 3 | gold.sla_compliance | equipment_type × region_name × month_year |
| 4 | gold.maintenance_summary | rig_id × month_year |
| 5 | gold.regional_benchmarks | region_name × month_year |

In [0]:
-- CELL 1 — SETUP
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
-- CELL 2 — CREATE GOLD TABLES (one-time setup)
-- Uses CREATE TABLE IF NOT EXISTS — safe to re-run, never drops data
-- Must exist before MERGE statements run

CREATE TABLE IF NOT EXISTS workspace.gold.rig_performance_summary (
    rig_id                      STRING,
    month_year                  TIMESTAMP,
    rig_name                    STRING,
    region_name                 STRING,
    total_records               BIGINT,
    avg_drilling_hours          DOUBLE,
    total_downtime_hours        DOUBLE,
    total_npt_hours             DOUBLE,
    avg_rop_ft_hr               DOUBLE,
    avg_cost_per_day            DOUBLE,
    avg_uptime_pct              DOUBLE,
    avg_downtime_pct            DOUBLE,
    avg_npt_pct                 DOUBLE,
    sla_met_count               BIGINT,
    sla_breach_count            BIGINT,
    sla_compliance_pct          DOUBLE,
    controllable_downtime_count BIGINT,
    total_incidents             BIGINT,
    serious_incidents           BIGINT,
    recordable_incidents        BIGINT,
    incident_downtime_hrs       DOUBLE,
    total_work_orders           BIGINT,
    planned_wo                  BIGINT,
    unplanned_wo                BIGINT,
    preventive_wo               BIGINT,
    high_cost_wo                BIGINT,
    unplanned_maint_pct         DOUBLE,
    total_maint_cost            DOUBLE,
    avg_cost_per_wo             DOUBLE,
    total_maint_hrs             DOUBLE,
    distinct_crew_count         BIGINT,
    avg_experience_years        DOUBLE,
    avg_hours_worked            DOUBLE,
    total_overtime_hours        DOUBLE,
    npt_crew_count              BIGINT,
    loaded_at                   TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.gold.equipment_reliability (
    equipment_type                  STRING,
    region_name                     STRING,
    month_year                      TIMESTAMP,
    active_rig_count                BIGINT,
    total_events                    BIGINT,
    failure_count                   BIGINT,
    downtime_event_count            BIGINT,
    total_downtime_hours            DOUBLE,
    avg_downtime_per_event          DOUBLE,
    avg_resolution_hrs              DOUBLE,
    avg_downtime_to_resolution_pct  DOUBLE,
    total_possible_hours            DOUBLE,
    availability_pct                DOUBLE,
    mtbf_hours                      DOUBLE,
    failure_rate_per_100h           DOUBLE,
    sla_compliance_pct              DOUBLE,
    sla_met_count                   BIGINT,
    sla_breach_count                BIGINT,
    loaded_at                       TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.gold.sla_compliance (
    equipment_type                  STRING,
    region_name                     STRING,
    month_year                      TIMESTAMP,
    total_events                    BIGINT,
    sla_met_count                   BIGINT,
    sla_breach_count                BIGINT,
    sla_compliance_pct              DOUBLE,
    failure_count                   BIGINT,
    avg_resolution_hrs              DOUBLE,
    total_resolution_hrs            DOUBLE,
    avg_downtime_to_resolution_pct  DOUBLE,
    total_work_orders               BIGINT,
    planned_wo                      BIGINT,
    unplanned_wo                    BIGINT,
    preventive_wo                   BIGINT,
    high_cost_wo                    BIGINT,
    planned_pct                     DOUBLE,
    total_cost                      DOUBLE,
    avg_cost_per_wo                 DOUBLE,
    avg_duration_hrs                DOUBLE,
    avg_cost_per_hour               DOUBLE,
    loaded_at                       TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.gold.maintenance_summary (
    rig_id                      STRING,
    month_year                  TIMESTAMP,
    rig_name                    STRING,
    region_name                 STRING,
    distinct_equipment_types    BIGINT,
    total_work_orders           BIGINT,
    planned_wo                  BIGINT,
    unplanned_wo                BIGINT,
    preventive_wo               BIGINT,
    high_cost_wo                BIGINT,
    long_duration_wo            BIGINT,
    total_cost                  DOUBLE,
    avg_cost_per_wo             DOUBLE,
    max_cost_single_wo          DOUBLE,
    avg_cost_per_hour           DOUBLE,
    total_duration_hrs          DOUBLE,
    avg_duration_hrs            DOUBLE,
    avg_technician_count        DOUBLE,
    unplanned_pct               DOUBLE,
    schedule_adherence_pct      DOUBLE,
    loaded_at                   TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.gold.regional_benchmarks (
    region_name                 STRING,
    month_year                  TIMESTAMP,
    active_rig_count            BIGINT,
    total_records               BIGINT,
    avg_uptime_pct              DOUBLE,
    avg_downtime_pct            DOUBLE,
    avg_npt_pct                 DOUBLE,
    avg_rop_ft_hr               DOUBLE,
    avg_cost_per_day            DOUBLE,
    total_incidents             BIGINT,
    serious_incidents           BIGINT,
    recordable_incidents        BIGINT,
    incident_rate_per_rig       DOUBLE,
    serious_incident_pct        DOUBLE,
    total_work_orders           BIGINT,
    unplanned_work_orders       BIGINT,
    unplanned_maint_pct         DOUBLE,
    total_maint_cost            DOUBLE,
    cost_per_rig                DOUBLE,
    total_sla_breaches          BIGINT,
    total_sla_met               BIGINT,
    sla_compliance_pct          DOUBLE,
    avg_resolution_hrs          DOUBLE,
    avg_availability_pct        DOUBLE,
    total_downtime_hours        DOUBLE,
    total_failures              BIGINT,
    score_uptime                DOUBLE,
    score_safety                DOUBLE,
    score_sla                   DOUBLE,
    score_availability          DOUBLE,
    composite_score             DOUBLE,
    performance_rank            BIGINT,
    loaded_at                   TIMESTAMP
) USING DELTA;

SELECT 'Gold tables ready' AS status;

status
Gold tables ready


In [0]:
-- CELL 3 — gold.rig_performance_summary
-- Grain   : rig_id × month_year
-- Sources : silver_rig_ops + silver_incident_log
--           + silver_maintenance_log + silver_crew_hours
-- Watermark: MAX(transformed_at) from silver_rig_ops

MERGE INTO workspace.gold.rig_performance_summary AS target
USING (
    WITH watermark AS (
        SELECT COALESCE(MAX(loaded_at), CAST('1970-01-01' AS TIMESTAMP)) AS wm
        FROM workspace.gold.rig_performance_summary
    ),
    affected_months AS (
        SELECT DISTINCT DATE_TRUNC('month', operation_date) AS month_year
        FROM workspace.silver.silver_rig_ops
        WHERE transformed_at > (SELECT wm FROM watermark)
        UNION
        SELECT DISTINCT DATE_TRUNC('month', incident_date)
        FROM workspace.silver.silver_incident_log
        WHERE transformed_at > (SELECT wm FROM watermark)
        UNION
        SELECT DISTINCT DATE_TRUNC('month', maintenance_date)
        FROM workspace.silver.silver_maintenance_log
        WHERE transformed_at > (SELECT wm FROM watermark)
        UNION
        SELECT DISTINCT DATE_TRUNC('month', work_date)
        FROM workspace.silver.silver_crew_hours
        WHERE transformed_at > (SELECT wm FROM watermark)
    ),
    rig_agg AS (
        SELECT
            rig_id,
            rig_name,
            region_name,
            DATE_TRUNC('month', operation_date)             AS month_year,
            COUNT(*)                                        AS total_records,
            ROUND(AVG(drilling_hours_clean), 2)             AS avg_drilling_hours,
            ROUND(SUM(downtime_hours), 2)                   AS total_downtime_hours,
            ROUND(SUM(npt_hours), 2)                        AS total_npt_hours,
            ROUND(AVG(rop_ft_hr_clean), 2)                  AS avg_rop_ft_hr,
            ROUND(AVG(cost_per_day_clean), 2)               AS avg_cost_per_day,
            ROUND(AVG(uptime_pct), 2)                       AS avg_uptime_pct,
            ROUND(AVG(downtime_pct), 2)                     AS avg_downtime_pct,
            ROUND(AVG(npt_pct), 2)                          AS avg_npt_pct,
            SUM(sla_met)                                    AS sla_met_count,
            SUM(is_sla_breach)                              AS sla_breach_count,
            SUM(CAST(is_controllable_downtime AS INT))      AS controllable_downtime_count
        FROM workspace.silver.silver_rig_ops
        WHERE is_valid = 1
        AND DATE_TRUNC('month', operation_date) IN (SELECT month_year FROM affected_months)
        GROUP BY rig_id, rig_name, region_name, DATE_TRUNC('month', operation_date)
    ),
    inc_agg AS (
        SELECT
            rig_id,
            DATE_TRUNC('month', incident_date)              AS month_year,
            COUNT(*)                                        AS total_incidents,
            SUM(is_serious_incident)                        AS serious_incidents,
            SUM(is_recordable)                              AS recordable_incidents,
            ROUND(SUM(downtime_caused_hrs), 2)              AS incident_downtime_hrs
        FROM workspace.silver.silver_incident_log
        WHERE is_valid = 1
        AND DATE_TRUNC('month', incident_date) IN (SELECT month_year FROM affected_months)
        GROUP BY rig_id, DATE_TRUNC('month', incident_date)
    ),
    maint_agg AS (
        SELECT
            rig_id,
            DATE_TRUNC('month', maintenance_date)           AS month_year,
            COUNT(*)                                        AS total_work_orders,
            SUM(CAST(is_planned AS INT))                    AS planned_wo,
            SUM(CASE WHEN is_planned = false THEN 1 ELSE 0 END) AS unplanned_wo,
            SUM(is_preventive)                              AS preventive_wo,
            SUM(is_high_cost)                               AS high_cost_wo,
            ROUND(SUM(cost), 2)                             AS total_maint_cost,
            ROUND(AVG(cost), 2)                             AS avg_cost_per_wo,
            ROUND(SUM(duration_hrs), 2)                     AS total_maint_hrs
        FROM workspace.silver.silver_maintenance_log
        WHERE is_valid = 1
        AND DATE_TRUNC('month', maintenance_date) IN (SELECT month_year FROM affected_months)
        GROUP BY rig_id, DATE_TRUNC('month', maintenance_date)
    ),
    crew_agg AS (
        SELECT
            rig_id,
            DATE_TRUNC('month', work_date)                  AS month_year,
            COUNT(DISTINCT crew_id)                         AS distinct_crew_count,
            ROUND(AVG(years_experience), 1)                 AS avg_experience_years,
            ROUND(AVG(hours_worked), 2)                     AS avg_hours_worked,
            ROUND(SUM(overtime_hours), 2)                   AS total_overtime_hours,
            SUM(is_npt_crew)                                AS npt_crew_count
        FROM workspace.silver.silver_crew_hours
        WHERE is_valid = 1
        AND DATE_TRUNC('month', work_date) IN (SELECT month_year FROM affected_months)
        GROUP BY rig_id, DATE_TRUNC('month', work_date)
    )
    SELECT
        r.rig_id,
        r.month_year,
        r.rig_name,
        r.region_name,
        r.total_records,
        r.avg_drilling_hours,
        r.total_downtime_hours,
        r.total_npt_hours,
        r.avg_rop_ft_hr,
        r.avg_cost_per_day,
        r.avg_uptime_pct,
        r.avg_downtime_pct,
        r.avg_npt_pct,
        r.sla_met_count,
        r.sla_breach_count,
        ROUND(
            CASE WHEN (r.sla_met_count + r.sla_breach_count) > 0
            THEN r.sla_met_count / (r.sla_met_count + r.sla_breach_count) * 100
            ELSE NULL END, 2)                               AS sla_compliance_pct,
        r.controllable_downtime_count,
        COALESCE(i.total_incidents, 0)                      AS total_incidents,
        COALESCE(i.serious_incidents, 0)                    AS serious_incidents,
        COALESCE(i.recordable_incidents, 0)                 AS recordable_incidents,
        COALESCE(i.incident_downtime_hrs, 0.0)              AS incident_downtime_hrs,
        COALESCE(m.total_work_orders, 0)                    AS total_work_orders,
        COALESCE(m.planned_wo, 0)                           AS planned_wo,
        COALESCE(m.unplanned_wo, 0)                         AS unplanned_wo,
        COALESCE(m.preventive_wo, 0)                        AS preventive_wo,
        COALESCE(m.high_cost_wo, 0)                         AS high_cost_wo,
        ROUND(
            CASE WHEN COALESCE(m.total_work_orders, 0) > 0
            THEN COALESCE(m.unplanned_wo, 0) / m.total_work_orders * 100
            ELSE 0 END, 2)                                  AS unplanned_maint_pct,
        COALESCE(m.total_maint_cost, 0.0)                   AS total_maint_cost,
        COALESCE(m.avg_cost_per_wo, 0.0)                    AS avg_cost_per_wo,
        COALESCE(m.total_maint_hrs, 0.0)                    AS total_maint_hrs,
        COALESCE(c.distinct_crew_count, 0)                  AS distinct_crew_count,
        c.avg_experience_years,
        c.avg_hours_worked,
        COALESCE(c.total_overtime_hours, 0.0)               AS total_overtime_hours,
        COALESCE(c.npt_crew_count, 0)                       AS npt_crew_count,
        current_timestamp()                                 AS loaded_at
    FROM rig_agg r
    LEFT JOIN inc_agg   i ON r.rig_id = i.rig_id AND r.month_year = i.month_year
    LEFT JOIN maint_agg m ON r.rig_id = m.rig_id AND r.month_year = m.month_year
    LEFT JOIN crew_agg  c ON r.rig_id = c.rig_id AND r.month_year = c.month_year
) AS source
ON  target.rig_id     = source.rig_id
AND target.month_year = source.month_year
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
580,0,0,580


In [0]:
SELECT 'rig_performance_summary' AS table_name, COUNT(*) AS row_count
FROM workspace.gold.rig_performance_summary;

table_name,row_count
rig_performance_summary,580


In [0]:
-- CELL 4 — gold.equipment_reliability
-- Grain  : equipment_type × region_name × month_year
-- Source : silver_equipment_events

MERGE INTO workspace.gold.equipment_reliability AS target
USING (
    WITH watermark AS (
        SELECT COALESCE(MAX(loaded_at), CAST('1970-01-01' AS TIMESTAMP)) AS wm
        FROM workspace.gold.equipment_reliability
    ),
    affected_months AS (
        SELECT DISTINCT DATE_TRUNC('month', event_date) AS month_year
        FROM workspace.silver.silver_equipment_events
        WHERE transformed_at > (SELECT wm FROM watermark)
    ),
    ev_agg AS (
        SELECT
            equipment_type,
            region_name,
            DATE_TRUNC('month', event_date)                 AS month_year,
            COUNT(DISTINCT rig_id)                          AS active_rig_count,
            COUNT(*)                                        AS total_events,
            SUM(failure_flag)                               AS failure_count,
            SUM(CASE WHEN downtime_caused_hrs > 0
                THEN 1 ELSE 0 END)                          AS downtime_event_count,
            ROUND(SUM(downtime_caused_hrs), 2)              AS total_downtime_hours,
            ROUND(AVG(CASE WHEN downtime_caused_hrs > 0
                THEN downtime_caused_hrs END), 2)           AS avg_downtime_per_event,
            ROUND(AVG(resolution_hrs), 2)                   AS avg_resolution_hrs,
            SUM(sla_met)                                    AS sla_met_count,
            SUM(is_sla_breach)                              AS sla_breach_count,
            ROUND(AVG(downtime_to_resolution_pct), 2)       AS avg_downtime_to_resolution_pct
        FROM workspace.silver.silver_equipment_events
        WHERE is_valid = 1
        AND DATE_TRUNC('month', event_date) IN (SELECT month_year FROM affected_months)
        GROUP BY equipment_type, region_name, DATE_TRUNC('month', event_date)
    )
    SELECT
        equipment_type,
        region_name,
        month_year,
        active_rig_count,
        total_events,
        failure_count,
        downtime_event_count,
        total_downtime_hours,
        COALESCE(avg_downtime_per_event, 0.0)               AS avg_downtime_per_event,
        avg_resolution_hrs,
        COALESCE(avg_downtime_to_resolution_pct, 0.0)       AS avg_downtime_to_resolution_pct,
        ROUND(active_rig_count
            * DAYOFMONTH(LAST_DAY(month_year)) * 24, 2)     AS total_possible_hours,
        ROUND((1 - total_downtime_hours /
            NULLIF(active_rig_count
                * DAYOFMONTH(LAST_DAY(month_year)) * 24, 0)
            ) * 100, 2)                                     AS availability_pct,
        ROUND(
            CASE WHEN failure_count > 0
            THEN (active_rig_count * DAYOFMONTH(LAST_DAY(month_year)) * 24
                - total_downtime_hours) / failure_count
            ELSE NULL END, 2)                               AS mtbf_hours,
        ROUND(
            CASE WHEN active_rig_count * DAYOFMONTH(LAST_DAY(month_year)) * 24 > 0
            THEN failure_count /
                (active_rig_count * DAYOFMONTH(LAST_DAY(month_year)) * 24) * 100
            ELSE 0 END, 4)                                  AS failure_rate_per_100h,
        ROUND(
            CASE WHEN (sla_met_count + sla_breach_count) > 0
            THEN sla_met_count / (sla_met_count + sla_breach_count) * 100
            ELSE NULL END, 2)                               AS sla_compliance_pct,
        COALESCE(sla_met_count, 0)                          AS sla_met_count,
        COALESCE(sla_breach_count, 0)                       AS sla_breach_count,
        current_timestamp()                                 AS loaded_at
    FROM ev_agg
) AS source
ON  target.equipment_type = source.equipment_type
AND target.region_name    = source.region_name
AND target.month_year     = source.month_year
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
725,0,0,725


In [0]:
SELECT 'equipment_reliability' AS table_name, COUNT(*) AS row_count
FROM workspace.gold.equipment_reliability;

table_name,row_count
equipment_reliability,725


In [0]:
-- CELL 5 — gold.sla_compliance
-- Grain  : equipment_type × region_name × month_year
-- Sources: silver_equipment_events (SLA) + silver_maintenance_log (type mix)

MERGE INTO workspace.gold.sla_compliance AS target
USING (
    WITH watermark AS (
        SELECT COALESCE(MAX(loaded_at), CAST('1970-01-01' AS TIMESTAMP)) AS wm
        FROM workspace.gold.sla_compliance
    ),
    affected_months AS (
        SELECT DISTINCT DATE_TRUNC('month', event_date) AS month_year
        FROM workspace.silver.silver_equipment_events
        WHERE transformed_at > (SELECT wm FROM watermark)
        UNION
        SELECT DISTINCT DATE_TRUNC('month', maintenance_date)
        FROM workspace.silver.silver_maintenance_log
        WHERE transformed_at > (SELECT wm FROM watermark)
    ),
    sla_agg AS (
        SELECT
            equipment_type,
            region_name,
            DATE_TRUNC('month', event_date)                 AS month_year,
            COUNT(*)                                        AS total_events,
            SUM(sla_met)                                    AS sla_met_count,
            SUM(is_sla_breach)                              AS sla_breach_count,
            SUM(failure_flag)                               AS failure_count,
            ROUND(AVG(resolution_hrs), 2)                   AS avg_resolution_hrs,
            ROUND(SUM(resolution_hrs), 2)                   AS total_resolution_hrs,
            ROUND(AVG(downtime_to_resolution_pct), 2)       AS avg_downtime_to_resolution_pct
        FROM workspace.silver.silver_equipment_events
        WHERE is_valid = 1
        AND DATE_TRUNC('month', event_date) IN (SELECT month_year FROM affected_months)
        GROUP BY equipment_type, region_name, DATE_TRUNC('month', event_date)
    ),
    mix_agg AS (
        SELECT
            equipment_type,
            region_name,
            DATE_TRUNC('month', maintenance_date)           AS month_year,
            COUNT(*)                                        AS total_work_orders,
            SUM(CAST(is_planned AS INT))                    AS planned_wo,
            SUM(CASE WHEN is_planned = false THEN 1 ELSE 0 END) AS unplanned_wo,
            SUM(is_preventive)                              AS preventive_wo,
            SUM(is_high_cost)                               AS high_cost_wo,
            ROUND(SUM(cost), 2)                             AS total_cost,
            ROUND(AVG(cost), 2)                             AS avg_cost_per_wo,
            ROUND(AVG(duration_hrs), 2)                     AS avg_duration_hrs,
            ROUND(AVG(cost_per_hour), 2)                    AS avg_cost_per_hour
        FROM workspace.silver.silver_maintenance_log
        WHERE is_valid = 1
        AND DATE_TRUNC('month', maintenance_date) IN (SELECT month_year FROM affected_months)
        GROUP BY equipment_type, region_name, DATE_TRUNC('month', maintenance_date)
    )
    SELECT
        s.equipment_type,
        s.region_name,
        s.month_year,
        s.total_events,
        COALESCE(s.sla_met_count, 0)                        AS sla_met_count,
        COALESCE(s.sla_breach_count, 0)                     AS sla_breach_count,
        ROUND(
            CASE WHEN (s.sla_met_count + s.sla_breach_count) > 0
            THEN s.sla_met_count / (s.sla_met_count + s.sla_breach_count) * 100
            ELSE NULL END, 2)                               AS sla_compliance_pct,
        s.failure_count,
        s.avg_resolution_hrs,
        s.total_resolution_hrs,
        COALESCE(s.avg_downtime_to_resolution_pct, 0.0)     AS avg_downtime_to_resolution_pct,
        COALESCE(m.total_work_orders, 0)                    AS total_work_orders,
        COALESCE(m.planned_wo, 0)                           AS planned_wo,
        COALESCE(m.unplanned_wo, 0)                         AS unplanned_wo,
        COALESCE(m.preventive_wo, 0)                        AS preventive_wo,
        COALESCE(m.high_cost_wo, 0)                         AS high_cost_wo,
        ROUND(
            CASE WHEN COALESCE(m.total_work_orders, 0) > 0
            THEN COALESCE(m.planned_wo, 0) / m.total_work_orders * 100
            ELSE 0 END, 2)                                  AS planned_pct,
        COALESCE(m.total_cost, 0.0)                         AS total_cost,
        COALESCE(m.avg_cost_per_wo, 0.0)                    AS avg_cost_per_wo,
        COALESCE(m.avg_duration_hrs, 0.0)                   AS avg_duration_hrs,
        COALESCE(m.avg_cost_per_hour, 0.0)                  AS avg_cost_per_hour,
        current_timestamp()                                 AS loaded_at
    FROM sla_agg s
    LEFT JOIN mix_agg m
        ON  s.equipment_type = m.equipment_type
        AND s.region_name    = m.region_name
        AND s.month_year     = m.month_year
) AS source
ON  target.equipment_type = source.equipment_type
AND target.region_name    = source.region_name
AND target.month_year     = source.month_year
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
725,0,0,725


In [0]:
SELECT 'sla_compliance' AS table_name, COUNT(*) AS row_count
FROM workspace.gold.sla_compliance;

table_name,row_count
sla_compliance,725


In [0]:
-- CELL 6 — gold.maintenance_summary
-- Grain  : rig_id × month_year
-- Source : silver_maintenance_log

MERGE INTO workspace.gold.maintenance_summary AS target
USING (
    WITH watermark AS (
        SELECT COALESCE(MAX(loaded_at), CAST('1970-01-01' AS TIMESTAMP)) AS wm
        FROM workspace.gold.maintenance_summary
    ),
    affected_months AS (
        SELECT DISTINCT DATE_TRUNC('month', maintenance_date) AS month_year
        FROM workspace.silver.silver_maintenance_log
        WHERE transformed_at > (SELECT wm FROM watermark)
    )
    SELECT
        rig_id,
        DATE_TRUNC('month', maintenance_date)               AS month_year,
        rig_name,
        region_name,
        COUNT(DISTINCT equipment_type)                      AS distinct_equipment_types,
        COUNT(*)                                            AS total_work_orders,
        SUM(CAST(is_planned AS INT))                        AS planned_wo,
        SUM(CASE WHEN is_planned = false THEN 1 ELSE 0 END) AS unplanned_wo,
        SUM(is_preventive)                                  AS preventive_wo,
        SUM(is_high_cost)                                   AS high_cost_wo,
        SUM(is_long_maintenance)                            AS long_duration_wo,
        ROUND(SUM(cost), 2)                                 AS total_cost,
        ROUND(AVG(cost), 2)                                 AS avg_cost_per_wo,
        ROUND(MAX(cost), 2)                                 AS max_cost_single_wo,
        ROUND(AVG(cost_per_hour), 2)                        AS avg_cost_per_hour,
        ROUND(SUM(duration_hrs), 2)                         AS total_duration_hrs,
        ROUND(AVG(duration_hrs), 2)                         AS avg_duration_hrs,
        ROUND(AVG(technician_count), 1)                     AS avg_technician_count,
        ROUND(
            CASE WHEN COUNT(*) > 0
            THEN SUM(CASE WHEN is_planned = false THEN 1 ELSE 0 END)
                 / COUNT(*) * 100
            ELSE 0 END, 2)                                  AS unplanned_pct,
        ROUND(
            CASE WHEN (SUM(CAST(is_planned AS INT)) +
                       SUM(CASE WHEN is_planned = false THEN 1 ELSE 0 END)) > 0
            THEN SUM(CAST(is_planned AS INT)) /
                 (SUM(CAST(is_planned AS INT)) +
                  SUM(CASE WHEN is_planned = false THEN 1 ELSE 0 END)) * 100
            ELSE NULL END, 2)                               AS schedule_adherence_pct,
        current_timestamp()                                 AS loaded_at
    FROM workspace.silver.silver_maintenance_log
    WHERE is_valid = 1
    AND DATE_TRUNC('month', maintenance_date) IN (SELECT month_year FROM affected_months)
    GROUP BY rig_id, rig_name, region_name, DATE_TRUNC('month', maintenance_date)
) AS source
ON  target.rig_id     = source.rig_id
AND target.month_year = source.month_year
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
580,0,0,580


In [0]:
SELECT 'maintenance_summary' AS table_name, COUNT(*) AS row_count
FROM workspace.gold.maintenance_summary;

table_name,row_count
maintenance_summary,580


In [0]:
-- CELL 7 — gold.regional_benchmarks
-- Grain  : region_name × month_year
-- Source : reads from gold tables 1-4 (already built this run)
-- Composite score: 40% uptime + 20% safety + 20% SLA + 20% availability
-- Watermark: anchored to silver transformed_at to avoid same-run timing issue

MERGE INTO workspace.gold.regional_benchmarks AS target
USING (
    WITH watermark AS (
        SELECT COALESCE(MAX(loaded_at), CAST('1970-01-01' AS TIMESTAMP)) AS wm
        FROM workspace.gold.regional_benchmarks
    ),
    -- Anchor to silver transformed_at — avoids same-run timing issue
    -- where gold loaded_at >= gold loaded_at on second run
    affected_months AS (
        SELECT DISTINCT DATE_TRUNC('month', operation_date) AS month_year
        FROM workspace.silver.silver_rig_ops
        WHERE transformed_at > (SELECT wm FROM watermark)
        UNION
        SELECT DISTINCT DATE_TRUNC('month', event_date)
        FROM workspace.silver.silver_equipment_events
        WHERE transformed_at > (SELECT wm FROM watermark)
        UNION
        SELECT DISTINCT DATE_TRUNC('month', maintenance_date)
        FROM workspace.silver.silver_maintenance_log
        WHERE transformed_at > (SELECT wm FROM watermark)
    ),
    region_ops AS (
        SELECT
            region_name,
            month_year,
            COUNT(DISTINCT rig_id)                          AS active_rig_count,
            SUM(total_records)                              AS total_records,
            ROUND(AVG(avg_uptime_pct), 2)                   AS avg_uptime_pct,
            ROUND(AVG(avg_downtime_pct), 2)                 AS avg_downtime_pct,
            ROUND(AVG(avg_npt_pct), 2)                      AS avg_npt_pct,
            ROUND(AVG(avg_rop_ft_hr), 2)                    AS avg_rop_ft_hr,
            ROUND(AVG(avg_cost_per_day), 2)                 AS avg_cost_per_day,
            SUM(total_incidents)                            AS total_incidents,
            SUM(serious_incidents)                          AS serious_incidents,
            SUM(recordable_incidents)                       AS recordable_incidents,
            SUM(total_work_orders)                          AS total_work_orders,
            SUM(unplanned_wo)                               AS unplanned_work_orders,
            ROUND(SUM(total_maint_cost), 2)                 AS total_maint_cost,
            SUM(sla_breach_count)                           AS total_sla_breaches,
            SUM(sla_met_count)                              AS total_sla_met
        FROM workspace.gold.rig_performance_summary
        WHERE month_year IN (SELECT month_year FROM affected_months)
        GROUP BY region_name, month_year
    ),
    region_sla AS (
        SELECT
            region_name,
            month_year,
            ROUND(AVG(sla_compliance_pct), 2)               AS sla_compliance_pct,
            ROUND(AVG(avg_resolution_hrs), 2)               AS avg_resolution_hrs
        FROM workspace.gold.sla_compliance
        WHERE month_year IN (SELECT month_year FROM affected_months)
        GROUP BY region_name, month_year
    ),
    region_equip AS (
        SELECT
            region_name,
            month_year,
            ROUND(AVG(availability_pct), 2)                 AS avg_availability_pct,
            ROUND(SUM(total_downtime_hours), 2)             AS total_downtime_hours,
            SUM(failure_count)                              AS total_failures
        FROM workspace.gold.equipment_reliability
        WHERE month_year IN (SELECT month_year FROM affected_months)
        GROUP BY region_name, month_year
    ),
    -- Compute composite score once — reused in both SELECT and RANK()
    scored AS (
        SELECT
            o.region_name,
            o.month_year,
            o.active_rig_count,
            o.total_records,
            o.avg_uptime_pct,
            o.avg_downtime_pct,
            o.avg_npt_pct,
            o.avg_rop_ft_hr,
            o.avg_cost_per_day,
            o.total_incidents,
            o.serious_incidents,
            o.recordable_incidents,
            ROUND(o.total_incidents / NULLIF(o.active_rig_count, 0), 3)
                                                            AS incident_rate_per_rig,
            ROUND(
                CASE WHEN o.total_incidents > 0
                THEN o.serious_incidents / o.total_incidents * 100
                ELSE 0 END, 2)                              AS serious_incident_pct,
            o.total_work_orders,
            o.unplanned_work_orders,
            ROUND(
                CASE WHEN o.total_work_orders > 0
                THEN o.unplanned_work_orders / o.total_work_orders * 100
                ELSE 0 END, 2)                              AS unplanned_maint_pct,
            o.total_maint_cost,
            ROUND(o.total_maint_cost / NULLIF(o.active_rig_count, 0), 2)
                                                            AS cost_per_rig,
            o.total_sla_breaches,
            o.total_sla_met,
            COALESCE(s.sla_compliance_pct, 0.0)             AS sla_compliance_pct,
            COALESCE(s.avg_resolution_hrs, 0.0)             AS avg_resolution_hrs,
            COALESCE(e.avg_availability_pct, 0.0)           AS avg_availability_pct,
            COALESCE(e.total_downtime_hours, 0.0)           AS total_downtime_hours,
            COALESCE(e.total_failures, 0)                   AS total_failures,
            -- Score components
            COALESCE(o.avg_uptime_pct, 0.0)                 AS score_uptime,
            ROUND(100 - COALESCE(
                CASE WHEN o.total_incidents > 0
                THEN o.serious_incidents / o.total_incidents * 100
                ELSE 0 END, 0.0), 2)                        AS score_safety,
            COALESCE(s.sla_compliance_pct, 0.0)             AS score_sla,
            COALESCE(e.avg_availability_pct, 0.0)           AS score_availability,
            -- Composite score computed once
            ROUND(
                COALESCE(o.avg_uptime_pct, 0.0)       * 0.40 +
                ROUND(100 - COALESCE(
                    CASE WHEN o.total_incidents > 0
                    THEN o.serious_incidents / o.total_incidents * 100
                    ELSE 0 END, 0.0), 2)               * 0.20 +
                COALESCE(s.sla_compliance_pct, 0.0)   * 0.20 +
                COALESCE(e.avg_availability_pct, 0.0)  * 0.20,
            2)                                              AS composite_score
        FROM region_ops o
        LEFT JOIN region_sla   s ON o.region_name = s.region_name AND o.month_year = s.month_year
        LEFT JOIN region_equip e ON o.region_name = e.region_name AND o.month_year = e.month_year
    )
    SELECT
        *,
        -- RANK references composite_score from the CTE — no duplication
        RANK() OVER (
            PARTITION BY month_year
            ORDER BY composite_score DESC
        )                                                   AS performance_rank,
        current_timestamp()                                 AS loaded_at
    FROM scored
) AS source
ON  target.region_name = source.region_name
AND target.month_year  = source.month_year
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
145,0,0,145


In [0]:
SELECT 'regional_benchmarks' AS table_name, COUNT(*) AS row_count
FROM workspace.gold.regional_benchmarks;

table_name,row_count
regional_benchmarks,145


In [0]:
-- CELL 8 — FINAL VALIDATION
-- Row counts across all gold tables

SELECT 'rig_performance_summary' AS table_name, COUNT(*) AS row_count
FROM workspace.gold.rig_performance_summary
UNION ALL
SELECT 'equipment_reliability',   COUNT(*)
FROM workspace.gold.equipment_reliability
UNION ALL
SELECT 'sla_compliance',          COUNT(*)
FROM workspace.gold.sla_compliance
UNION ALL
SELECT 'maintenance_summary',     COUNT(*)
FROM workspace.gold.maintenance_summary
UNION ALL
SELECT 'regional_benchmarks',     COUNT(*)
FROM workspace.gold.regional_benchmarks
ORDER BY table_name;

table_name,row_count
equipment_reliability,725
maintenance_summary,580
regional_benchmarks,145
rig_performance_summary,580
sla_compliance,725


In [0]:
-- Regional benchmarks latest month — performance rankings
SELECT
    performance_rank,
    region_name,
    composite_score,
    score_uptime,
    score_safety,
    score_sla,
    score_availability,
    avg_uptime_pct,
    incident_rate_per_rig,
    sla_compliance_pct,
    avg_availability_pct
FROM workspace.gold.regional_benchmarks
WHERE month_year = (SELECT MAX(month_year) FROM workspace.gold.regional_benchmarks)
ORDER BY performance_rank;

performance_rank,region_name,composite_score,score_uptime,score_safety,score_sla,score_availability,avg_uptime_pct,incident_rate_per_rig,sla_compliance_pct,avg_availability_pct
1,Permian,94.74,91.76,92.86,98.17,99.16,91.76,3.5,98.17,99.16
2,Bakken,92.54,82.04,100.0,99.17,99.44,82.04,2.25,99.17,99.44
3,Eagle Ford,92.47,88.24,90.0,97.17,98.7,88.24,2.5,97.17,98.7
4,Marcellus,90.56,77.66,100.0,98.33,99.17,77.66,1.5,98.33,99.17
5,Gulf,87.81,85.27,72.73,97.0,98.8,85.27,2.75,97.0,98.8


In [0]:
-- Top 10 rigs latest month
SELECT
    rig_name,
    region_name,
    month_year,
    avg_uptime_pct,
    avg_rop_ft_hr,
    avg_cost_per_day,
    total_incidents,
    serious_incidents,
    total_work_orders,
    unplanned_maint_pct,
    total_maint_cost
FROM workspace.gold.rig_performance_summary
WHERE month_year = (SELECT MAX(month_year) FROM workspace.gold.rig_performance_summary)
ORDER BY avg_uptime_pct DESC, total_incidents ASC
LIMIT 10;

rig_name,region_name,month_year,avg_uptime_pct,avg_rop_ft_hr,avg_cost_per_day,total_incidents,serious_incidents,total_work_orders,unplanned_maint_pct,total_maint_cost
Rig 004,Permian,2026-05-01T00:00:00.000Z,94.21,83.44,28201.22,6,1,14,0.0,91122.79
Rig 001,Permian,2026-05-01T00:00:00.000Z,92.05,83.88,28200.84,4,0,20,0.0,159143.97
Rig 003,Permian,2026-05-01T00:00:00.000Z,90.52,84.02,28388.17,4,0,15,0.0,102984.47
Rig 002,Permian,2026-05-01T00:00:00.000Z,90.26,82.91,27849.92,0,0,20,0.0,192237.0
Rig 005,Eagle Ford,2026-05-01T00:00:00.000Z,88.76,77.22,29987.06,3,1,7,0.0,41734.58
Rig 007,Eagle Ford,2026-05-01T00:00:00.000Z,88.44,74.9,30220.87,1,0,18,0.0,137106.33
Rig 006,Eagle Ford,2026-05-01T00:00:00.000Z,88.23,75.2,29462.8,2,0,20,0.0,123528.95
Rig 008,Eagle Ford,2026-05-01T00:00:00.000Z,87.53,74.92,29838.15,4,0,19,0.0,142783.46
Rig 018,Gulf,2026-05-01T00:00:00.000Z,86.98,68.11,44793.05,4,2,11,0.0,89575.44
Rig 020,Gulf,2026-05-01T00:00:00.000Z,85.51,68.38,44715.28,1,0,17,0.0,130431.14
